# 04 - Integrated Operational Silver
### Goal
- join market prices, weather, and grid events silver tables
- create a unified operational view
- write integrated silver Delta table

In [0]:
%run ../01_setup/00_config

In [0]:
from pyspark.sql import functions as F

In [0]:
market_prices_df = spark.table(silver_market_prices_table)
weather_df       = spark.table(silver_weather_table)
grid_events_df   = spark.table(silver_grid_events_table)
asset_reference_df = spark.table(bronze_asset_reference_table)
event_type_reference_df = spark.table(bronze_event_type_reference_table)
region_reference_df = spark.table(bronze_region_reference_table)

print(f"Market prices rows: {market_prices_df.count()}")
print(f"Weather rows:       {weather_df.count()}")
print(f"Grid events rows:   {grid_events_df.count()}")
print(f"Asset reference rows: {asset_reference_df.count()}")
print(f"Event type reference rows: {event_type_reference_df.count()}")
print(f"Region reference rows: {region_reference_df.count()}")

## Join on event_date and region

In [0]:
market_prices_agg_df = (
    market_prices_df
    .groupBy("event_date", "region")
    .agg(
        F.round(F.avg("price_eur_mwh"), 2).alias("avg_price_eur_mwh"),
        F.round(F.sum("volume_mwh"), 2).alias("total_volume_mwh"),
        F.max("is_high_price").alias("has_high_price"),
        F.first("source_system").alias("source_system")
    )
)

weather_agg_df = (
    weather_df
    .groupBy("event_date", "region")
    .agg(
        F.round(F.avg("temperature_c"), 2).alias("avg_temperature_c"),
        F.round(F.avg("wind_speed_kmh"), 2).alias("avg_wind_speed_kmh"),
        F.round(F.sum("precipitation_mm"), 2).alias("total_precipitation_mm"),
        F.max("is_severe_weather").alias("has_severe_weather"),
        F.first("wind_class").alias("wind_class")
    )
)

print("Market prices aggregated:", market_prices_agg_df.count())
print("Weather aggregated:", weather_agg_df.count())

In [0]:
grid_events_enriched_df = (
    grid_events_df
    .join(
        asset_reference_df.select("asset_id", "asset_name", "asset_type"),
        on="asset_id",
        how="left"
    )
    .join(
        event_type_reference_df.select("event_type", "category"),
        on="event_type",
        how="left"
    )
)

region_reference_df_clean = region_reference_df.select("region", "country_code", "operating_zone")

integrated_df = (
    market_prices_agg_df
    .join(
        weather_agg_df.select(
            "event_date", "region",
            "avg_temperature_c", "avg_wind_speed_kmh",
            "total_precipitation_mm", 
            "has_severe_weather", "wind_class"
        ),
        on=["event_date", "region"],
        how="left"
    )
    .join(
        grid_events_enriched_df.select(
            "event_date", "region",
            "event_id", "event_type", "severity",
            "duration_minutes", "is_critical_event", "duration_band",
            "asset_id", "asset_name", "asset_type", "category"
        ),
        on=["event_date", "region"],
        how="left"
    )
    .join(
        region_reference_df_clean,
        on="region",
        how="left"
    )
)

display(integrated_df.limit(10))
print(f"Integrated rows: {integrated_df.count()}")